In [ ]:
# # !pip install datasets==4.0
# # !pip install --upgrade datasets transformers huggingface_hub
# # !rm -rf /root/.cache/huggingface/datasets/
# !pip install -U bitsandbytes

In [ ]:
from transformers import (
    AutoTokenizer, # language models
    AutoModelForCausalLM,
    Trainer, # fine-tuning
    TrainingArguments,
    DataCollatorForLanguageModeling, # part of pipeline responsible for assembling
    BitsAndBytesConfig, # BitsAndBytes for quantum compression, less data size for models, yet not losing in speed at all
    )
from sklearn.model_selection import train_test_split
import torch # pyTorch
import safetensors.torch
from datasets import Dataset # converting text file in dataset (increases file reading speed)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model # (Parameter Efficient Fine Tuning) -> we take LoRA only
import pandas as pd

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class LLMLoaderPipeline:

  model_name: str
  file_path: str
  test_size: float = 0.1
  max_length: int = 512

  model: Optional[AutoModelForCausalLM] = None
  tokenizer: Optional[AutoTokenizer] = None
  dataset: Optional[Dataset] = None
  train_dataset: Optional[Dataset] = None
  val_dataset: Optional[Dataset] = None

  def __post_init__(self):
    self.load_model()
    self.file_read_pandas()
    self.tokenize_and_split()

  def load_model(self):
    self.model = AutoModelForCausalLM.from_pretrained(
      self.model_name,
      torch_dtype=torch.float16, ## eat less resources
      device_map="auto",
      quantization_config=BitsAndBytesConfig(load_in_8bit=True) # bitsAndBytes
    )
    self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
    self.tokenizer.pad_token = self.tokenizer.eos_token # in case some custom models dont have pad_token by default

  def file_read_pandas(self):
    df = pd.read_csv(self.file_path)
    print(df.info())
    df = df.dropna()
    df = df.drop_duplicates()

    def create_text_row(row):
        return f"Greek: {row['greek_word']} | Cypriot: {row['cypriot_word']} | English: {row['english_definition']}"

    df['text'] = df.apply(create_text_row, axis=1)

    self.dataset = Dataset.from_dict({"text": df['text'].tolist()})

  def tokenize_and_split(self):
    def token_func(batch):
        return self.tokenizer(
            batch["text"], # list demand
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
        )

    split_dataset = self.dataset.train_test_split(
        test_size=self.test_size,
        seed=42,
        shuffle=True
    )

    self.train_dataset = split_dataset["train"].map(token_func, batched=True)
    self.val_dataset = split_dataset["test"].map(token_func, batched=True)

  def get_model_and_tokenizer(self):
      return self.model, self.tokenizer

  def get_datasets(self):
      return self.train_dataset, self.val_dataset

In [ ]:
pipeline = LLMLoaderPipeline(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    file_path="./dictionary.csv",
)

In [ ]:
model, tokenizer = pipeline.get_model_and_tokenizer()
train_dataset, val_dataset = pipeline.get_datasets()

In [ ]:
train_dataset[0]

In [ ]:
# !pip install trl

from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
      r=8, # a rank, the bigger the rank, more accuracy we get, but becomes slower
      lora_alpha=42, # an influence of LoRA on a model
      target_modules=["q_proj", "v_proj"], # there are modules we touch to change, q_proj = query, v_proj = value
      lora_dropout=0.05, # in order to avoid overtraining
      bias="none",
      task_type="CAUSAL_LM" # model wise
    )

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

In [ ]:
import wandb
# from tqdm import tqdm

wandb.login()

In [ ]:
# from trl import SFTTrainer, SFTConfig

# wandb.init(project="llama-small", name="greek-finetuning")

# trainer = SFTTrainer(
#     model = model,
#     train_dataset = train_dataset,
#     eval_dataset = val_dataset,
#     args = SFTConfig(

#         # MAIN SETTINGS
#         ######################
#         dataset_text_field = "text",
#         per_device_train_batch_size = 1,
#         gradient_accumulation_steps = 4,
#         warmup_steps = 5,
#         max_steps = 100,
#         disable_tqdm = False,
#         learning_rate = 1e-6,
#         logging_steps = 1,
#         optim = "adamw_torch",
#         weight_decay = 0.01,
#         lr_scheduler_type = "linear",
#         seed = 1337,
#         ###################

#         # LOGGING
#         #####################
#         report_to = "wandb",
#         log_level = "info",
#         ###################

#         #####################
#         #### CPU SETTINGS
#         bf16 = False,
#         fp16 = False,
#         dataloader_pin_memory = False,
#         dataloader_num_workers = 0,
#         no_cuda = True,
#         #####################
#     ),
# )

# trainer.train()

# wandb.finish()

In [ ]:
@dataclass
class LoraTrainerPipeline:

  model: AutoModelForCausalLM
  tokenizer: AutoTokenizer
  train_dataset: Dataset
  val_dataset: Optional[Dataset] = None
  output_dir: str = "./tiny-llama"

  max_steps: int = 100
  batch_size: int = 10
  learning_rate: float = 5e-6

  lora_model: Optional[AutoModelForCausalLM] = None
  trainer: Optional[Trainer] = None

  def lora_training(self):
    model = prepare_model_for_kbit_training(self.model)
    lora_config = LoraConfig(
      r=8, # a rank, the bigger the rank, more accuracy we get, but becomes slower
      lora_alpha=42, # an influence of LoRA on a model
      target_modules=["q_proj", "v_proj"], # there are modules we touch to change, q_proj = query, v_proj = value
      lora_dropout=0.05, # in order to avoid overtraining
      bias="none",
      task_type="CAUSAL_LM" # model wise
    )
    self.lora_model = get_peft_model(model, lora_config)
    # model.gradient_checkpointing_enable()

  def model_train(self, max_steps=None, batch_size=None, learning_rate=None):

    max_steps = max_steps or self.max_steps
    batch_size = batch_size or self.batch_size
    learning_rate = learning_rate or self.learning_rate

    self.lora_training()

    data_collator = DataCollatorForLanguageModeling(
      tokenizer=self.tokenizer,
      mlm=False
    )

    # adam is working under the hood by default
    training_args = TrainingArguments(
      output_dir=self.output_dir,
      overwrite_output_dir=True,
      max_steps=max_steps,
      per_device_train_batch_size=batch_size, # количество рассмотренных обьектов за один раз -> усреднение -> лучшая точность
      save_steps=50,
      save_total_limit=1,
      report_to="wandb",
      prediction_loss_only=True,
      fp16=True,
      learning_rate=learning_rate,
      ######################
      logging_steps=10,    # <- training losses
      ######################
      eval_strategy="steps",
      eval_steps=10, # <- validation losses
      ######################
    )

     #use_cache=False, ## turn off cache to avoid cuda errors TODO

    self.trainer = Trainer(
      model=self.lora_model,
      args=training_args,
      train_dataset=self.train_dataset,  # ← training
      eval_dataset=self.val_dataset,     # ← validations
      tokenizer=self.tokenizer,
      data_collator=data_collator,
    )

    self.trainer.train()

  def merge_and_unload(self, checkpoint_path):
    print("Merging LoRA and unloading PEFT weights...")
    self.lora_model = PeftModel.from_pretrained(self.lora_model, checkpoint_path)
    self.lora_model = self.lora_model.merge_and_unload()
    return self.lora_model


In [15]:
max_steps = 100

##########################
pipeline = LoraTrainerPipeline(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    max_steps=100,
    batch_size=1,
    learning_rate=1e-6
)
##########################

wandb.init(project="llama-small", name="greek-finetuning")
pipeline.model_train(max_steps=max_steps) # <- LoRA fine-tuning
wandb.finish()


KeyboardInterrupt: 